In [1]:
# 📦 Install if not already done
#!pip install "langchain==1.3.11" langchain-openai langchain-community langchain-text-splitters faiss-cpu tiktoken python-dotenv langgraph



JupyterLab


Python 3 (ipykernel)
# 📦 Install if not already done
# !pip install "langchain==1.3.11" langchain-openai langchain-community langchain-text-splitters faiss-cpu tiktoken python-dotenv langgraph
LangChain RAG Retriever Types — Rewritten for LangChain v1 (langchain==1.3.11)
This notebook keeps the same scenario as the original 10)LangChain_RAG_Types.ipynb: a reference table of LangChain's retriever types, followed by a working example that builds a VectorStoreRetriever over sample.txt, wraps it as an agent tool, adds conversational memory, and asks the same 3 questions.

Every import path in the table below was verified by actually installing langchain==1.3.11 and inspecting the package (not guessed from memory) — see the last column.

Retriever Type	Purpose	Ideal Usage Scenario	Verified langchain==1.3.11 import
VectorStoreRetriever	Retrieve documents based on vector similarity search (e.g. FAISS, Chroma, Pinecone)	Basic RAG setup for fetching semantically relevant documents	Not a separate class — call .as_retriever() on any vector store, e.g. langchain_community.vectorstores.FAISS
ContextualCompressionRetriever	Compresses retrieved docs using an LLM to return only the most relevant parts	Large context docs (PDFs, transcripts) where only a portion of each doc is relevant	from langchain_classic.retrievers import ContextualCompressionRetriever (requires pip install langchain-classic)
MultiQueryRetriever	Generates multiple rephrased queries to improve retrieval coverage	Ambiguous or variably-phrased queries; improves diversity of retrieved documents	from langchain_classic.retrievers import MultiQueryRetriever
ParentDocumentRetriever	Retrieves a full parent document instead of the split chunk	Full document context required (blog, article, contract analysis)	from langchain_classic.retrievers import ParentDocumentRetriever
BM25Retriever	Keyword-based retriever using the classic BM25 algorithm	Exact keyword matches (legal, medical, or short precise-language docs)	from langchain_community.retrievers import BM25Retriever (requires pip install langchain-community rank_bm25)
EnsembleRetriever	Combines multiple retrievers (e.g. vector + BM25) with weighted scoring	Merge semantic + keyword search for hybrid performance	from langchain_classic.retrievers import EnsembleRetriever
TimeWeightedVectorStoreRetriever	VectorStoreRetriever + time-based decay to prefer recent documents	Chatbot memory or news search where recency matters	from langchain_classic.retrievers import TimeWeightedVectorStoreRetriever
TavilySearchAPIRetriever	Uses Tavily's web search API to retrieve live web data	Real-world events, breaking news, info outside your local vector DB	from langchain_community.retrievers import TavilySearchAPIRetriever — or, the actively-recommended v1 approach, the TavilySearch tool from langchain_tavily, passed straight into create_agent
Where things live now, verified against the installed package:

langchain==1.3.11 itself only ships agents, chat_models, embeddings, messages, rate_limiters, tools — no retrievers, vectorstores, chains, memory, or text_splitter module anymore.
All 7 non-VectorStoreRetriever classes above now live in langchain-classic (the legacy-code package split out for v1) or langchain-community. They still import and work, but neither package is part of the actively-developed langchain core going forward, and langchain-community specifically is in sunset/maintenance mode (its own deprecation warning says so on import). Prefer a purpose-built integration package where one exists (e.g. langchain_tavily instead of TavilySearchAPIRetriever).
Migration cheat-sheet for the worked example below
Legacy piece	v1 replacement
ConversationalRetrievalChain.from_llm(llm, retriever) wrapped in a StructuredTool	A plain @tool function that calls retriever.invoke(query) and has the LLM answer from that context
ConversationBufferMemory(memory_key="chat_history", ...)	checkpointer=InMemorySaver() + thread_id
initialize_agent(..., AgentType.CONVERSATIONAL_REACT_DESCRIPTION, handle_parsing_errors=True)	create_agent(model, tools, checkpointer=...)
langchain.chat_models.ChatOpenAI / langchain.embeddings.OpenAIEmbeddings	langchain_openai.ChatOpenAI / langchain_openai.OpenAIEmbeddings
langchain.vectorstores.FAISS	langchain_community.vectorstores.FAISS
langchain.text_splitter.CharacterTextSplitter	langchain_text_splitters.CharacterTextSplitter
One correctness note on the original code: rag_tool_fn always called rag_chain.invoke({"question": q, "chat_history": []}) — passing an empty chat history on every single call. So ConversationalRetrievalChain's own memory was never actually used; only the outer agent's ConversationBufferMemory carried context between turns. The rewrite below has a single memory system (the agent's checkpointer) instead of two disconnected ones, which removes that redundancy — and the bug — entirely.

Reference: https://docs.langchain.com/oss/python/releases/langchain-v1#create_agent

In [2]:
# 1) VectorStoreRetriever — vector-based, embedding similarity search, general-purpose RAG

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import CharacterTextSplitter
from langgraph.checkpoint.memory import InMemorySaver

import os
from dotenv import load_dotenv

# 2️⃣ Load API keys
load_dotenv(".env")
os.environ["OPENAI_API_KEY"] = "Openai_api_Key"

# NOTE: the original notebook relied on the default model (gpt-3.5-turbo era),
# which has since been retired — pin an explicit, currently-supported model.
llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

# 3. Vector DB
with open("sample.txt", "r", encoding="utf-8") as f:
    text_data = f.read()

# 🧠 Split the text into smaller chunks
splitter = CharacterTextSplitter(separator="\n", chunk_size=300, chunk_overlap=50)
texts = splitter.split_text(text_data)

embedding = OpenAIEmbeddings()
vectorstore = FAISS.from_texts(texts, embedding)
retriever = vectorstore.as_retriever()

# 4. Wrap the retriever as an agent tool (replaces ConversationalRetrievalChain + StructuredTool)
@tool
def RAG_QA(question: str) -> str:
    """Use this to answer questions about LangChain."""
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    answer = llm.invoke(
        f"Answer the question using only the context below.\n\nContext:\n{context}\n\nQuestion: {question}"
    )
    return answer.content

# 5. Memory for chat history (checkpointer, replaces ConversationBufferMemory)
checkpointer = InMemorySaver()
thread_config = {"configurable": {"thread_id": "rag-types-demo-1"}}

# 6. Create agent (replaces initialize_agent)
agent = create_agent(
    model=llm,
    tools=[RAG_QA],
    checkpointer=checkpointer,
)

# 7. Run conversation
print("1️⃣ First question")
res1 = agent.invoke({"messages": [{"role": "user", "content": "What is LangChain?"}]}, thread_config)
print("Answer:", res1["messages"][-1].content)

print("\n2️⃣ Follow-up")
res2 = agent.invoke({"messages": [{"role": "user", "content": "Who created it?"}]}, thread_config)
print("Answer:", res2["messages"][-1].content)

print("\n3️⃣ Ask again")
res3 = agent.invoke(
    {"messages": [{"role": "user", "content": "Explain LangChain again simply."}]}, thread_config
)
print("Answer:", res3["messages"][-1].content)

C:\Users\KAVITHA\AppData\Local\Temp\ipykernel_25856\1721462350.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


1️⃣ First question
Answer: LangChain is a framework designed for building applications that utilize large language models (LLMs). It provides tools and components to facilitate the development of applications leveraging the capabilities of LLMs. If you want, I can provide more detailed information about its features and use cases.

2️⃣ Follow-up
Answer: LangChain was created by Harrison Chase. If you want to know more about the creator or the history of LangChain, feel free to ask!

3️⃣ Ask again
Answer: LangChain is a tool that helps developers build apps using powerful language models like ChatGPT. It makes it easier to connect these models with other data and tools, so the apps can do more useful things like answering questions, summarizing text, or having conversations.
